# 3. Microsoft Sentinel and Defender XDR

## SIEM vs SOAR

| | SIEM | SOAR |
|-|------|------|
| **Stands for** | Security Information and Event Management | Security Orchestration, Automation, and Response |
| **Purpose** | Collect, analyze, and correlate security logs | Automate incident response workflows |
| **Example action** | "Alert: suspicious login from new country" | "Automatically disable user account + create ticket" |
| **Microsoft tool** | Microsoft Sentinel | Microsoft Sentinel (both in one!) |

**Microsoft Sentinel is both SIEM *and* SOAR** — it collects logs AND automates responses.

## Microsoft Sentinel

Cloud-native SIEM/SOAR built on Azure. Ingests data from everywhere:

| Data source | Connector |
|------------|----------|
| Microsoft 365 | Built-in |
| Azure resources | Built-in (diagnostics) |
| Entra ID sign-in/audit | Built-in |
| Defender XDR incidents | Built-in |
| Firewalls (Palo Alto, Fortinet...) | Data connector |
| Linux syslog | Log Analytics agent |
| Custom apps | CEF, Syslog, REST API |

### The four Sentinel capabilities

| Capability | What it does |
|-----------|-------------|
| **Collect** | Ingest data from any source at cloud scale |
| **Detect** | Find threats with built-in analytics rules and ML |
| **Investigate** | Explore incidents with interactive graphs and timelines |
| **Respond** | Automate with playbooks (Logic Apps) |

---
## KQL — the query language of Sentinel

Sentinel stores data in **Log Analytics workspaces** and queries it with **KQL (Kusto Query Language)**. KQL is also used by Defender, Azure Monitor, and Azure Data Explorer.

Here are some common Sentinel queries (you won't write KQL on the SC-900 exam, but understanding it helps):

In [ ]:
# We'll simulate KQL queries against mock log data
from datetime import datetime, timedelta
import random
import json

# Generate mock sign-in logs (like Entra ID SigninLogs table)
USERS = ['alice@contoso.com', 'bob@contoso.com', 'carol@contoso.com', 'attacker@evil.com']
LOCATIONS = ['Seattle', 'London', 'Moscow', 'Beijing', 'Office']
STATUSES = ['Success', 'Failure']

random.seed(42)
SIGN_IN_LOGS = []
now = datetime.now()
for i in range(50):
    user = random.choice(USERS)
    loc = random.choice(LOCATIONS)
    status = 'Success' if random.random() > 0.3 else 'Failure'
    if user == 'attacker@evil.com':
        status = 'Failure'
        loc = random.choice(['Moscow', 'Beijing'])
    SIGN_IN_LOGS.append({
        'TimeGenerated': (now - timedelta(hours=random.randint(0, 48))).isoformat()[:19],
        'UserPrincipalName': user,
        'Location': loc,
        'ResultType': status,
        'AppDisplayName': random.choice(['Azure Portal', 'Outlook', 'Teams']),
    })

print(f'Generated {len(SIGN_IN_LOGS)} mock sign-in log entries')
print(f'Sample: {json.dumps(SIGN_IN_LOGS[0], indent=2)}')

In [ ]:
# KQL simulation: Find failed sign-ins by user
# Real KQL: SigninLogs | where ResultType != 'Success' | summarize count() by UserPrincipalName
print('=== KQL: Failed sign-ins by user ===')
print('    SigninLogs | where ResultType != "Success" | summarize count() by UserPrincipalName\n')

from collections import Counter
failed = Counter(log['UserPrincipalName'] for log in SIGN_IN_LOGS if log['ResultType'] == 'Failure')
for user, count in failed.most_common():
    alert = ' ⚠️ SUSPICIOUS — high failure count!' if count > 5 else ''
    print(f'  {user:<30} {count} failures{alert}')

In [ ]:
# KQL simulation: Logins from unusual locations
# Real KQL: SigninLogs | where Location !in ('Seattle', 'Office', 'London') | project TimeGenerated, UserPrincipalName, Location
print('=== KQL: Sign-ins from unusual locations ===')
print('    SigninLogs | where Location !in ("Seattle", "Office", "London")\n')

trusted_locations = {'Seattle', 'Office', 'London'}
suspicious = [log for log in SIGN_IN_LOGS if log['Location'] not in trusted_locations]
for log in suspicious[:10]:
    print(f'  {log["TimeGenerated"]}  {log["UserPrincipalName"]:<30}  {log["Location"]:<10}  {log["ResultType"]}')

In [ ]:
# KQL simulation: Brute force detection (>3 failures in 1 hour from same user)
print('=== KQL: Brute force detection (>3 failures per user in 1 hour) ===')
print('    SigninLogs | where ResultType == "Failure"')
print('    | summarize FailCount=count() by UserPrincipalName, bin(TimeGenerated, 1h)')
print('    | where FailCount > 3\n')

# Group failures by user and hour
from collections import defaultdict
hourly_failures = defaultdict(int)
for log in SIGN_IN_LOGS:
    if log['ResultType'] == 'Failure':
        hour = log['TimeGenerated'][:13]  # group by hour
        hourly_failures[(log['UserPrincipalName'], hour)] += 1

brute_force = {k: v for k, v in hourly_failures.items() if v > 3}
if brute_force:
    for (user, hour), count in brute_force.items():
        print(f'  🚨 {user} had {count} failures at {hour}:00 — possible brute force!')
else:
    print('  No brute force patterns detected in this sample.')
    print('  (In real Sentinel, analytics rules run continuously and create incidents automatically.)')

### Sentinel automation: Playbooks

When an analytics rule fires, Sentinel can trigger a **playbook** (Azure Logic App) to automate the response:

```
Analytics rule triggers → Incident created → Playbook runs:
  1. Block user account in Entra ID
  2. Send alert to Teams channel
  3. Create ServiceNow ticket
  4. Enrich incident with threat intelligence
```

This is the **SOAR** part of Sentinel.

---
## Microsoft Defender XDR

**XDR = eXtended Detection and Response**. It's a suite of Defender products that share signals and provide unified threat detection across your entire environment.

| Defender product | What it protects |
|-----------------|-------------------|
| **Defender for Endpoint** | Devices (laptops, servers) — EDR, vulnerability management |
| **Defender for Office 365** | Email and collaboration — anti-phishing, safe attachments, safe links |
| **Defender for Identity** | On-prem AD — detects lateral movement, pass-the-hash, reconnaissance |
| **Defender for Cloud Apps** | SaaS apps — shadow IT discovery, app governance, session controls |
| **Defender Vulnerability Management** | Discover and remediate vulnerabilities across your estate |
| **Defender Threat Intelligence** | Threat articles, IOCs, attacker profiles |

### The Microsoft Defender portal

All Defender XDR products are managed from **security.microsoft.com** — a unified portal for:
- Incidents (correlated alerts across products)
- Threat hunting (KQL across all data)
- Action center (approve/deny automated actions)
- Secure Score (for Microsoft 365)

### How XDR correlates alerts

Traditional approach: each product sends its own alerts. You get 50 alerts for one attack.

XDR approach: Defender correlates related alerts into a single **incident**:

In [ ]:
# Simulate XDR incident correlation
ALERTS = [
    {'source': 'Defender for Office 365', 'title': 'Phishing email delivered to alice@contoso.com', 'time': '09:00'},
    {'source': 'Defender for Endpoint',   'title': 'Suspicious PowerShell execution on ALICE-LAPTOP', 'time': '09:05'},
    {'source': 'Entra ID Protection',     'title': 'Atypical token usage for alice@contoso.com', 'time': '09:07'},
    {'source': 'Defender for Cloud Apps',  'title': 'Mass file download from SharePoint by alice@contoso.com', 'time': '09:10'},
    {'source': 'Defender for Identity',    'title': 'Lateral movement attempt from ALICE-LAPTOP', 'time': '09:15'},
]

print('=== Without XDR: 5 separate alerts ===\n')
for a in ALERTS:
    print(f'  [{a["time"]}] {a["source"]}: {a["title"]}')

print('\n=== With XDR: 1 correlated incident ===\n')
print('📋 Incident #4821: "Multi-stage attack on alice@contoso.com"')
print('   Severity: HIGH')
print('   Status: Active')
print('   Kill chain:')
print('   1. 📧 Initial access: phishing email (Office 365)')
print('   2. 💻 Execution: malicious PowerShell (Endpoint)')
print('   3. 🔑 Credential theft: token abuse (Entra ID)')
print('   4. 📁 Data exfiltration: mass download (Cloud Apps)')
print('   5. ↔️ Lateral movement: spread to other machines (Identity)')
print(f'   \n   Entities involved: alice@contoso.com, ALICE-LAPTOP, 3 SharePoint sites')
print(f'   Automated response: user account disabled, device isolated')

---
## Sentinel vs Defender XDR

| | Microsoft Sentinel | Defender XDR |
|-|-------------------|---------------|
| **Type** | SIEM/SOAR | XDR |
| **Scope** | Everything (any log source) | Microsoft 365 + endpoints + identity |
| **Data** | Logs, events, threat intel | Alerts from Defender products |
| **Best for** | SOC teams needing unified visibility across *all* sources | Automated detection within Microsoft ecosystem |
| **Portal** | Azure portal (Sentinel workspace) | security.microsoft.com |

They work **together**: Defender XDR incidents can flow into Sentinel for broader correlation with non-Microsoft data.

### Exam tip

- Sentinel = SIEM + SOAR, cloud-native, any data source.
- Defender XDR = automated threat detection across Microsoft products.
- Sentinel can ingest Defender XDR incidents.
- The unified Defender portal is at security.microsoft.com.

---
## Summary

| Concept | Key fact |
|---------|----------|
| **SIEM** | Collect and analyze security logs |
| **SOAR** | Automate incident response |
| **Sentinel** | Cloud-native SIEM + SOAR. Uses KQL. Playbooks = Logic Apps. |
| **Defender XDR** | Correlated threat detection across endpoint, email, identity, cloud apps |
| **Defender for Endpoint** | Device protection (EDR) |
| **Defender for Office 365** | Email protection (anti-phishing) |
| **Defender for Identity** | On-prem AD protection |
| **Defender for Cloud Apps** | SaaS protection (shadow IT, CASB) |

**Next lab**: [04 — Compliance and Purview](../../04-compliance-and-purview/)